# Somo la 02 - Kuchunguza Mfumo wa Wakala wa Microsoft

**Mfumo wa Wakala wa Microsoft (MAF)** ni mfumo uliounganishwa wa kujenga mawakala wa AI. Unatoa usanifu safi, unaoweza kusanidiwa na una vipengele vikuu vinne vya msingi:

- **Mteja** – huunganishwa na sehemu ya mfano wa AI na hushughulikia mawasiliano
- **Wakala** – huwaweka mteja pamoja na maagizo na ufafanuzi wa zana
- **Zana** – huongeza uwezo wa wakala kwa kutumia kazi maalum ambazo mfano unaweza kuitisha
- **Kikao** – huendeleza historia ya mazungumzo kwa mwingiliano wa mizunguko mingi

Katika somo hili, tutajenga **wakala wa uhifadhi wa safari** anayekagua upatikanaji wa marudio kwa kutumia dhana hizi.


## Usanidi


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Kuelewa Muundo wa Mfumo wa Wakala

Mfumo wa Wakala wa Microsoft unafuata muundo wa tabaka:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Mteja** – `AzureAIProjectAgentProvider` huungana na usambazaji wa Azure OpenAI. Inashughulikia uthibitishaji, uundaji wa ombi, na uchambuzi wa majibu.
2. **Wakala** – Imetengenezwa kutoka kwa mteja kupitia `provider.create_agent()`, wakala huunganisha upatikanaji wa mfano na maelekezo (akamisho la mfumo) na zana.
3. **Zana** – Vifunction vya Python vilivyowekwa alama na `@tool` ambavyo wakala anaweza kuita kutekeleza vitendo au kupata data.
4. **Kikao** – Kifaa cha `AgentSession` (kilichoundwa kupitia `agent.create_session()`) kinakumbatia historia ya mazungumzo, kuwezesha mazungumzo yenye mizunguko mingi ambapo wakala anakumbuka muktadha wa awali.

Tujenge kila tabaka hatua kwa hatua.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kuongeza Zana na Decorator @tool

Zana huruhusu mawakala kuchukua hatua zaidi ya kuzalisha maandishi. Decorator `@tool` hubadilisha kazi ya kawaida ya Python kuwa kitu ambacho wakala anaweza kuita.

Mambo muhimu:
- Tumia `Annotated[type, "description"]` ili mfano uelewe kila parameta.
- Docstring huwa maelezo ya zana ambayo mfano unaona.
- `approval_mode="never_require"` inamaanisha zana inatekelezwa moja kwa moja bila uthibitisho wa mtumiaji.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Kuunda Wakala na Zana

Sasa tunaunganisha mteja, maelekezo, na zana kuwa wakala. `maelekezo` hufanya kazi kama salamu ya mfumo — hufafanua utu na tabia ya wakala.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mazungumzo ya Mzunguko-Mwingi na Vikao

`AgentSession` (iliyoundwa kupitia `agent.create_session()`) huhifadhi ujumbe wote katika mazungumzo. Kwa kupitisha kikao kimoja kwa kila mwito wa `agent.run()`, wakala anaweza kupata historia yote ya mazungumzo na kurejelea ujumbe wa awali.

Tunapita `tools=[check_destination_availability]` ili wakala aweze kutumia kaguzi yetu ya upatikanaji kila mzunguko.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Muhtasari

Katika somo hili ulichunguza nguzo nne za Mfumo wa Wakala wa Microsoft:

| Dhana | Uliyojifunza |
|---------|------------------|
| **Mteja** | `AzureAIProjectAgentProvider` inaunganisha na Azure OpenAI kwa uthibitisho wa msingi wa vibali |
| **Wakala** | `provider.create_agent()` huunganisha muundo wa mfano na maelekezo na jina |
| **Vifaa** | Dekoreta ya `@tool` huonesha kazi za Python kwa wakala kupiga simu |
| **Kikao** | `agent.create_session()` hutoa historia ya mazungumzo katika mizunguko mingi |

Vitu hivi vya msingi huunganishwa pamoja kuunda mawakala wanaoweza kuendesha mazungumzo ya asili, kupiga simu kwa kazi za nje, na kudumisha muktadha — msingi wa mifumo ya wakala zilizoendelea katika masomo yajayo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kumbusho**:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za moja kwa moja zinaweza kuwa na makosa au upungufu. Nyaraka asilia katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo cha kuaminika. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na watu inashauriwa. Hatutawajibika kwa kuelewana vibaya au tafsiri zisizo sahihi zitakazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
